In [10]:
import warnings
warnings.filterwarnings(
    action="ignore")
warnings.filterwarnings("error", category=FutureWarning,
                        module=r"torch\.nn\.modules\.module")


In [ ]:
# exp_sgd_svm.py  ── plain-SGD, default SVM loss
import subprocess, sys, time, statistics, pandas as pd
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
BEST_CFG = {
    "Cornell"     : {"epochs":10, "lr_scale":0.1},
    "Dermatology" : {"epochs":10, "lr_scale":0.5},
    "HHAR"        : {"epochs":30, "lr_scale":0.1},
    "ISOLET"      : {"epochs":30, "lr_scale":2.0},
    "ORL"         : {"epochs":30, "lr_scale":0.1},
    "USPS"        : {"epochs":20, "lr_scale":0.5},
    "Vehicle"     : {"epochs":10, "lr_scale":1.0},
}

OPTIMIZER   = "sgd"
LOSS_ARG    = []                
EPSILONS    = [1.0, 2.0, 4.0, 8.0]
N_REPEATS   = 5
OUT_DIR     = Path("exp_results_sgd_svm"); OUT_DIR.mkdir(exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
def run_once(ds, eps, epc, lr, seed=None):
    cmd = [
        "python", "main-opacus.py",
        "--data", ds,
        "--optimizer", OPTIMIZER,
        "--num_epoch", str(epc),
        "--lr_scale", str(lr),
        "--eps", str(eps),
    ] + LOSS_ARG                      
    if seed is not None:
        cmd += ["--seed", str(seed)] 

    print(" ".join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    out, err = p.communicate()
    if err:  print(err, file=sys.stderr)

    acc = None
    for ln in out.splitlines():
        low = ln.lower()
        if "test accuracy" in low:
            tokens = [s for s in low.replace('%','').split() if s.replace('.','',1).isdigit()]
            if tokens:
                acc = float(tokens[-1])
                break
    if acc is None:
        raise RuntimeError(f"accuracy not found for {ds}, eps={eps}")
    return acc
# ──────────────────────────────────────────────────────────────────────────────
def main():
    rows = []
    for ds, cfg in BEST_CFG.items():
        for eps in EPSILONS:
            acc_list = []
            for rep in range(N_REPEATS):
                acc = run_once(ds, eps, cfg["epochs"], cfg["lr_scale"], seed=None)
                acc_list.append(acc)
                time.sleep(0.5)
            mean = statistics.mean(acc_list)
            std  = statistics.stdev(acc_list)
            rows.append({
                "dataset": ds,
                "epsilon": eps,
                "mean": mean,
                "std": std,
                "latex": f"{mean:.3f}$\\pm${std:.3f}"
            })
            pd.DataFrame(rows).to_csv(OUT_DIR/"summary_progress.csv", index=False)
            print(f"[{ds:11s} | ε={eps:>4}] {rows[-1]['latex']}")

    df = pd.DataFrame(rows)
    df.to_csv(OUT_DIR/"summary_final.csv", index=False)
    print("\n▶ Summary:", OUT_DIR/"summary_final.csv")

    # LaTeX 행 출력
    print("\n============= LaTeX Rows =============")
    for ds in BEST_CFG.keys():
        vals = df.query("dataset == @ds").sort_values("epsilon")["latex"].tolist()
        print(f"{ds} & " + " & ".join(vals) + r" \\")
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()


python main-opacus.py --data Cornell --optimizer sgd --num_epoch 10 --lr_scale 0.1 --eps 1.0


/home/user/micromamba/lib/python3.11/site-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)



python main-opacus.py --data Cornell --optimizer sgd --num_epoch 10 --lr_scale 0.1 --eps 1.0


/home/user/micromamba/lib/python3.11/site-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)



python main-opacus.py --data Cornell --optimizer sgd --num_epoch 10 --lr_scale 0.1 --eps 1.0


...

python main-opacus.py --data Vehicle --optimizer sgd --num_epoch 10 --lr_scale 1.0 --eps 8.0


/home/user/micromamba/lib/python3.11/site-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)



python main-opacus.py --data Vehicle --optimizer sgd --num_epoch 10 --lr_scale 1.0 --eps 8.0


/home/user/micromamba/lib/python3.11/site-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)



[Vehicle     | ε= 8.0] 0.589$\pm$0.064

▶ 최종 요약 저장: exp_results_sgd_svm/summary_final.csv

============= LaTeX Rows =============
Cornell & 0.637$\pm$0.030 & 0.688$\pm$0.018 & 0.681$\pm$0.000 & 0.681$\pm$0.000 \\
Dermatology & 0.489$\pm$0.160 & 0.638$\pm$0.138 & 0.603$\pm$0.046 & 0.570$\pm$0.123 \\
HHAR & 0.848$\pm$0.010 & 0.853$\pm$0.010 & 0.872$\pm$0.019 & 0.853$\pm$0.015 \\
ISOLET & 0.066$\pm$0.025 & 0.097$\pm$0.014 & 0.106$\pm$0.046 & 0.094$\pm$0.030 \\
ORL & 0.025$\pm$0.015 & 0.037$\pm$0.029 & 0.022$\pm$0.010 & 0.048$\pm$0.021 \\
USPS & 0.865$\pm$0.008 & 0.883$\pm$0.006 & 0.886$\pm$0.004 & 0.889$\pm$0.003 \\
Vehicle & 0.494$\pm$0.072 & 0.560$\pm$0.048 & 0.614$\pm$0.032 & 0.589$\pm$0.064 \\


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
exp_sgd_svm.py ── DP-SGD + SVM loss + (옵션) 선형 LR-decay
"""

import subprocess
import sys
import time
import re
import statistics
import pandas as pd
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
BEST_CFG = {
    "Cornell":     {"epochs": 10, "lr_scale": 0.1},
    "Dermatology": {"epochs": 10, "lr_scale": 0.5},
    "HHAR":        {"epochs": 30, "lr_scale": 0.1},
    "ISOLET":      {"epochs": 30, "lr_scale": 2.0},
    "ORL":         {"epochs": 30, "lr_scale": 0.1},
    "USPS":        {"epochs": 20, "lr_scale": 0.5},
    "Vehicle":     {"epochs": 10, "lr_scale": 1.0},
}

OPTIMIZER  = "sgd"
LOSS_ARG   = []                           #
EPSILONS   = [1.0, 2.0, 4.0, 8.0]
N_REPEATS  = 5
OUT_DIR    = Path("exp_results_sgd_svm")
OUT_DIR.mkdir(exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
def run_once(
    ds: str,
    eps: float,
    epc: int,
    lr: float,
    seed: int | None = None,
    use_lr_decay: bool = True,
) -> float:
    """main-opacus-decay.py """
    cmd = [
        "python", "main-opacus-decay.py",
        "--data",       ds,
        "--optimizer",  OPTIMIZER,
        "--num_epoch",  str(epc),
        "--lr_scale",   str(lr),
        "--eps",        str(eps),
    ] + LOSS_ARG

    if use_lr_decay:
        cmd.append("--lr_decay")

    if seed is not None:
        cmd += ["--seed", str(seed)]

    print(" ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True)
    out, err = proc.communicate()
    if err:
        print(err, file=sys.stderr)

    acc = None
    for ln in out.splitlines():
        m = re.search(r"\bacc=([0-9.]+)", ln)
        if m:
            acc = float(m.group(1))  
    if acc is None:
        raise RuntimeError(f"accuracy not found for {ds}, eps={eps}")
    return acc

# ──────────────────────────────────────────────────────────────────────────────
def main() -> None:
    rows = []
    for ds, cfg in BEST_CFG.items():
        for eps in EPSILONS:
            acc_list = []
            for rep in range(N_REPEATS):
                acc = run_once(ds, eps, cfg["epochs"], cfg["lr_scale"], seed=None)
                acc_list.append(acc)

            mean = statistics.mean(acc_list)
            std  = statistics.stdev(acc_list)
            rows.append({
                "dataset": ds,
                "epsilon": eps,
                "mean":    mean,
                "std":     std,
                "latex":   f"{mean:.3f}$\\pm${std:.3f}"
            })
            pd.DataFrame(rows).to_csv(OUT_DIR / "summary_progress.csv", index=False)
            print(f"[{ds:11s} | ε={eps:>4}] {rows[-1]['latex']}")

    df = pd.DataFrame(rows)
    df.to_csv(OUT_DIR / "summary_final.csv", index=False)
    print("\n▶ Summary:", OUT_DIR / "summary_final.csv")

    print("\n============= LaTeX Rows =============")
    for ds in BEST_CFG.keys():
        vals = df.query("dataset == @ds").sort_values("epsilon")["latex"].tolist()
        print(f"{ds} & " + " & ".join(vals) + r" \\")
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()


python main-opacus-decay.py --data Cornell --optimizer sgd --num_epoch 10 --lr_scale 0.1 --eps 1.0 --lr_decay


/home/user/micromamba/lib/python3.11/site-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)



python main-opacus-decay.py --data Cornell --optimizer sgd --num_epoch 10 --lr_scale 0.1 --eps 1.0 --lr_decay


/home/user/micromamba/lib/python3.11/site-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)



python main-opacus-decay.py --data Cornell --optimizer sgd --num_epoch 10 --lr_scale 0.1 --eps 1.0 --lr_decay


...

python main-opacus-decay.py --data Vehicle --optimizer sgd --num_epoch 10 --lr_scale 1.0 --eps 8.0 --lr_decay


/home/user/micromamba/lib/python3.11/site-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)



python main-opacus-decay.py --data Vehicle --optimizer sgd --num_epoch 10 --lr_scale 1.0 --eps 8.0 --lr_decay


/home/user/micromamba/lib/python3.11/site-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)



[Vehicle     | ε= 8.0] 0.752$\pm$0.011

▶ 최종 요약 저장: exp_results_sgd_svm/summary_final.csv

============= LaTeX Rows =============
Cornell & 0.654$\pm$0.014 & 0.726$\pm$0.028 & 0.754$\pm$0.010 & 0.767$\pm$0.009 \\
Dermatology & 0.892$\pm$0.063 & 0.941$\pm$0.031 & 0.938$\pm$0.015 & 0.959$\pm$0.019 \\
HHAR & 0.940$\pm$0.003 & 0.952$\pm$0.004 & 0.956$\pm$0.002 & 0.956$\pm$0.002 \\
ISOLET & 0.324$\pm$0.059 & 0.517$\pm$0.046 & 0.653$\pm$0.025 & 0.712$\pm$0.034 \\
ORL & 0.092$\pm$0.027 & 0.190$\pm$0.032 & 0.215$\pm$0.032 & 0.273$\pm$0.024 \\
USPS & 0.909$\pm$0.003 & 0.924$\pm$0.003 & 0.925$\pm$0.001 & 0.927$\pm$0.003 \\
Vehicle & 0.672$\pm$0.040 & 0.732$\pm$0.016 & 0.744$\pm$0.018 & 0.752$\pm$0.011 \\
